# Notebook 02 — Temporal Split & Task Definition

**Goal:** Build reproducible, leakage-safe, time-based datasets for evaluating next-window *view-item* retrieval. RetailRocket contains interactions, **not impressions**, so these labels do not measure CTR.

**Evaluation protocol:** At each fixed prediction timestamp, use only earlier events as user history. Future views within the following 14 days are positive targets (a set of items, not necessarily the single next view). Validation and test windows are separated in time. No randomly sampled negative pairs are created in this notebook.

**Important:** A model tuned on validation must not use test-period events. A final model for test can be refit on events strictly before the test timestamp.

## 1. Paths and imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
events_path = RAW_DIR / 'events.csv'
assert events_path.is_file(), f'Missing: {events_path} (cwd={cwd})'
print('Project root:', PROJECT_ROOT)
print('Input:', events_path)

Project root: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks
Input: C:\YR_Python_Code\multi-stage-ads-recommendation\notebooks\data\raw\events.csv


## 2. Load, validate and sort raw events

Do not replace the raw CSV. Sorting uses a stable algorithm; identical millisecond timestamps cannot establish the true within-millisecond event order.

In [2]:
events = pd.read_csv(events_path)
required = {'timestamp','visitorid','event','itemid','transactionid'}
assert set(events.columns) == required
assert events[list(required - {'transactionid'})].notna().all().all()
assert events['event'].isin(['view','addtocart','transaction']).all()
events['datetime'] = pd.to_datetime(events['timestamp'], unit='ms', utc=True, errors='coerce')
assert events['datetime'].notna().all()
events = events.sort_values('timestamp', kind='mergesort').reset_index(drop=True)
print('Rows:', len(events), 'Time:', events.datetime.min(), '→', events.datetime.max())
print('Exact duplicate rows (original five columns):', events[list(required)].duplicated().sum())
print(events.event.value_counts())

Rows: 2756101 Time: 2015-05-03 03:00:04.384000+00:00 → 2015-09-18 02:59:47.788000+00:00
Exact duplicate rows (original five columns): 460
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64


## 3. Define fixed-time evaluation windows

At **2015-08-01 00:00 UTC**, we make validation recommendations; future `view` events through August 14 are labels. At **2015-09-01 00:00 UTC**, we make test recommendations; future `view` events through September 14 are labels. Both are 14-day windows. The gap between windows is intentional. User history is frozen at each prediction timestamp.

In [3]:
VAL_START = pd.Timestamp('2015-08-01', tz='UTC')
VAL_END   = pd.Timestamp('2015-08-15', tz='UTC')  # exclusive
TEST_START = pd.Timestamp('2015-09-01', tz='UTC')
TEST_END   = pd.Timestamp('2015-09-15', tz='UTC')  # exclusive
assert events.datetime.min() < VAL_START < VAL_END <= TEST_START < TEST_END <= events.datetime.max()

train_history = events.loc[events.datetime < VAL_START].copy()
val_future = events.loc[(events.datetime >= VAL_START) & (events.datetime < VAL_END)].copy()
test_history = events.loc[events.datetime < TEST_START].copy()
test_future = events.loc[(events.datetime >= TEST_START) & (events.datetime < TEST_END)].copy()

for name, df in [('train_history',train_history),('val_future',val_future),
                 ('test_history',test_history),('test_future',test_future)]:
    print(f'{name:15s} {len(df):>10,} rows | {df.datetime.min()} → {df.datetime.max()}')
assert train_history.datetime.max() < VAL_START
assert test_history.datetime.max() < TEST_START
assert val_future.datetime.between(VAL_START, VAL_END, inclusive='left').all()
assert test_future.datetime.between(TEST_START, TEST_END, inclusive='left').all()

train_history    1,899,029 rows | 2015-05-03 03:00:04.384000+00:00 → 2015-07-31 23:59:58.354000+00:00
val_future         252,086 rows | 2015-08-01 00:00:14.822000+00:00 → 2015-08-14 23:59:56.673000+00:00
test_history     2,452,391 rows | 2015-05-03 03:00:04.384000+00:00 → 2015-08-31 23:59:58.647000+00:00
test_future        260,132 rows | 2015-09-01 00:00:02.005000+00:00 → 2015-09-14 23:59:45.566000+00:00


## 4. Build positive labels and eligible-user cohorts

Only future `view` events define positive retrieval targets in this baseline. We deduplicate `(visitorid,itemid)` **within each label window** so repeated views of the same item do not inflate Recall@K. Evaluation users must have at least one event in the corresponding pre-cutoff history. This is a warm-user evaluation; report the excluded cold-user cohort separately. We do not require a positive label for every eligible user, but standard Recall@K is defined only for users with at least one positive.

In [4]:
def make_labels(future_events, history_events):
    eligible_users = pd.Index(history_events['visitorid'].unique())
    labels_all = (future_events.loc[future_events.event.eq('view'), ['visitorid','itemid']]
                  .drop_duplicates().reset_index(drop=True))
    warm_mask = labels_all.visitorid.isin(eligible_users)
    labels_warm = labels_all.loc[warm_mask].reset_index(drop=True)
    return labels_all, labels_warm, eligible_users

val_labels_all, val_labels, val_users = make_labels(val_future, train_history)
test_labels_all, test_labels, test_users = make_labels(test_future, test_history)

for name, labels_all, labels_warm, eligible in [
    ('validation',val_labels_all,val_labels,val_users),
    ('test',test_labels_all,test_labels,test_users)]:
    print(f'{name}: {len(eligible):,} historical users; '
          f'{labels_all.visitorid.nunique():,} future-view users; '
          f'{labels_warm.visitorid.nunique():,} warm users with positives; '
          f'{len(labels_warm):,} warm positive user-item pairs')
    print('  cold future-view users:',
          labels_all.visitorid.nunique() - labels_warm.visitorid.nunique())

assert val_labels[['visitorid','itemid']].duplicated().sum() == 0
assert test_labels[['visitorid','itemid']].duplicated().sum() == 0

validation: 962,528 historical users; 145,763 future-view users; 13,329 warm users with positives; 28,634 warm positive user-item pairs
  cold future-view users: 132434
test: 1,250,856 historical users; 149,560 future-view users; 15,137 warm users with positives; 31,876 warm positive user-item pairs
  cold future-view users: 134423


## 5. Separate warm-item and new-item targets

Some future-view items have **never appeared before the cutoff**. A retrieval method that only indexes historical items cannot retrieve them. We keep all warm-user targets, but also publish a warm-item subset and item-cold-start diagnostics. Do not silently drop these labels when reporting results.

In [5]:
def item_cold_start_report(name, history, labels):
    known_items = pd.Index(history.itemid.unique())
    warm_item_mask = labels.itemid.isin(known_items)
    warm_item_labels = labels.loc[warm_item_mask].copy()
    print(f'{name}: warm-item positives {len(warm_item_labels):,}/{len(labels):,} '
          f'({warm_item_mask.mean():.2%} of positive pairs)' if len(labels) else f'{name}: no labels')
    return warm_item_labels

val_labels_warm_item = item_cold_start_report('validation', train_history, val_labels)
test_labels_warm_item = item_cold_start_report('test', test_history, test_labels)

validation: warm-item positives 26,155/28,634 (91.34% of positive pairs)
test: warm-item positives 29,274/31,876 (91.84% of positive pairs)


## 6. Leakage and consistency checks

Historical event counts, item popularity, and candidate indexes **must be recomputed separately** from `train_history` for validation and `test_history` for test. Do not reuse global EDA counts from Notebook 01.

In [6]:
assert train_history.timestamp.is_monotonic_increasing
assert test_history.timestamp.is_monotonic_increasing
assert not val_labels.empty and not test_labels.empty
assert val_labels.visitorid.isin(train_history.visitorid).all()
assert test_labels.visitorid.isin(test_history.visitorid).all()
assert val_labels_warm_item.itemid.isin(train_history.itemid).all()
assert test_labels_warm_item.itemid.isin(test_history.itemid).all()
assert VAL_END <= TEST_START
print('PASS: cutoff, cohort, label uniqueness, and warm-item checks')

PASS: cutoff, cohort, label uniqueness, and warm-item checks


## 7. Save reproducible artifacts

CSV is used for compatibility without optional Parquet dependencies. File names encode their intended purpose. All are **offline research artifacts**, not online serving features.

In [7]:
outputs = {
    'train_history.csv': train_history,
    'validation_future_events.csv': val_future,
    'test_history.csv': test_history,
    'test_future_events.csv': test_future,
    'validation_view_labels_all.csv': val_labels_all,
    'validation_view_labels_warm_users.csv': val_labels,
    'validation_view_labels_warm_users_items.csv': val_labels_warm_item,
    'test_view_labels_all.csv': test_labels_all,
    'test_view_labels_warm_users.csv': test_labels,
    'test_view_labels_warm_users_items.csv': test_labels_warm_item,
}
for filename, df in outputs.items():
    target = PROCESSED_DIR / filename
    df.to_csv(target, index=False)
    print(f'Saved {filename}: {len(df):,} rows')

Saved train_history.csv: 1,899,029 rows
Saved validation_future_events.csv: 252,086 rows
Saved test_history.csv: 2,452,391 rows
Saved test_future_events.csv: 260,132 rows
Saved validation_view_labels_all.csv: 200,935 rows
Saved validation_view_labels_warm_users.csv: 28,634 rows
Saved validation_view_labels_warm_users_items.csv: 26,155 rows
Saved test_view_labels_all.csv: 208,168 rows
Saved test_view_labels_warm_users.csv: 31,876 rows
Saved test_view_labels_warm_users_items.csv: 29,274 rows


## 8. Milestone summary and next notebook

- **Prediction timestamp:** validation Aug 1, test Sep 1 (UTC).
- **Labels:** unique items with future `view` interactions in the next 14 days.
- **Eligible users:** users observed before the relevant timestamp; evaluation recall is calculated for those with at least one future positive.
- **Warm-item subset:** separately available for retrieval methods unable to serve previously unseen items.
- **No explicit negatives or impression-based CTR:** these are unavailable in the data.
- **No future features:** train/validation retrieval indexes must be fit from `train_history`; final test indexes from `test_history` only.
- **Temporal overlap caveat:** test history includes the validation period; this is valid for a final model rebuilt as of Sep 1, **not** for tuning on test labels.

**Next:** `03_eda_user_item_analysis.ipynb` can analyze train-only distributions and cohorts. Notebook 04 will implement the first popularity retrieval baseline and evaluate Recall@K on these frozen labels. More advanced ranking label construction and negative sampling are deferred until ranking notebooks, where the candidate exposure assumptions must be made explicit.